# 01c · Perfiles de Holmberg por subhalo — análisis de un grupo

**Objetivo:** para un grupo FoF seleccionado, cargar **todos los subhalos** con `SubhaloFlag != 0` (sin corte de masa), ajustar el perfil de Holmberg a cada uno individualmente, y visualizar únicamente las partículas estelares que quedan **por fuera** de ese radio — la envolvente extendida de cada galaxia.

**Diferencia respecto a 01b_:**

| Notebook | Fuente | Por subhalo | Pregunta |
|----------|--------|-------------|----------|
| `01b_` | `loadHalo` + máscara | Un perfil global BCG+ICL difusa | ¿Cuánta masa está fuera del BCG? |
| `01c_` | `loadSubhalo` × N_sub | Un perfil Holmberg por galaxia | ¿Qué partículas están en la envolvente de cada galaxia? |

**Parámetro principal a cambiar:**
```python
GROUP_IDX = 0   # ← índice FoF local en el catálogo (posición en group_idx, no ID TNG)
```

In [1]:
import sys, os, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from astropy.cosmology import FlatLambdaCDM
from scipy.interpolate import interp1d
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, './original_shift_code')
import illustris_python as il
import Catalogue
import params_icl as P

FIG_PDF = './figuras/01c_perfil_subhalos_grupo/pdf'
FIG_PNG = './figuras/01c_perfil_subhalos_grupo/png'
os.makedirs(FIG_PDF, exist_ok=True)
os.makedirs(FIG_PNG, exist_ok=True)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 200,
                     'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)

## Selección del grupo a analizar

Cambia `GROUP_IDX` para seleccionar otro grupo del catálogo.  
`GROUP_IDX` es la **posición en el catálogo ICL** (0-based), no el ID FoF de TNG.

In [2]:
# ── Parámetro principal ───────────────────────────────────────────────────
GROUP_IDX = 0   # ← posición en el catálogo (0 = primer grupo del catálogo ICL)
# ─────────────────────────────────────────────────────────────────────────

# Cargar catálogo base
with h5py.File(P.CATALOG_OUT, 'r') as f:
    group_idx_cat = f['group_idx'][:]     # índices FoF absolutos TNG
    M200c_all     = f['M200c_Msun'][:]
    R200c_all     = f['R200c_kpc'][:]
    GroupPos_all  = f['GroupPos_kpc'][:]
    bcg_sub_all   = f['bcg_sub_idx'][:]

# Propiedades del grupo seleccionado
FOF_IDX  = int(group_idx_cat[GROUP_IDX])   # ID FoF absoluto en TNG
M200c    = M200c_all[GROUP_IDX]
R200c    = R200c_all[GROUP_IDX]
GroupPos = GroupPos_all[GROUP_IDX]          # posición del BCG [kpc]
BCG_sub  = int(bcg_sub_all[GROUP_IDX])      # subfind ID del subhalo central

Header = il.groupcat.loadHeader(P.basePath, P.SNAP)

print(f"Grupo seleccionado:")
print(f"  GROUP_IDX (catálogo)  : {GROUP_IDX}")
print(f"  FOF_IDX (TNG absoluto): {FOF_IDX}")
print(f"  BCG subfind ID        : {BCG_sub}")
print(f"  log M200c             : {np.log10(M200c):.2f} M_sun")
print(f"  R200c                 : {R200c:.1f} kpc")

Grupo seleccionado:
  GROUP_IDX (catálogo)  : 0
  FOF_IDX (TNG absoluto): 0
  BCG subfind ID        : 0
  log M200c             : 14.58 M_sun
  R200c                 : 1523.0 kpc


## Cargar subhalos del grupo

Obtenemos todos los subhalos que pertenecen a este FoF directamente desde el groupcat de TNG.  
Filtramos por `SubhaloFlag != 0` (excluye subhalos espurios detectados por Subfind dentro de discos/streams).  
**No aplicamos corte de masa** — queremos todas las galaxias resueltas.

In [3]:
fields_sub = ['SubhaloFlag', 'SubhaloPos', 'SubhaloMassType',
              'SubhaloMass', 'SubhaloGrNr', 'SubhaloHalfmassRadType']

# Carga global de todos los subhalos — cuando se pasan múltiples fields
# loadSubhalos devuelve un dict; con un solo field devuelve el array directamente
# (por eso el loop campo-por-campo fallaba al intentar tmp['NombreCampo'])
all_subs   = il.groupcat.loadSubhalos(P.basePath, P.SNAP, fields=fields_sub)
N_sub_total = len(all_subs['SubhaloGrNr'])
all_sub_ids = np.arange(N_sub_total, dtype=int)   # position IS the subfind ID

# Reshape guards — algunos campos pueden llegar aplanados
mtype = all_subs['SubhaloMassType']
if mtype.ndim == 1:
    mtype = mtype.reshape(-1, 6)
hmrad = all_subs['SubhaloHalfmassRadType']
if hmrad.ndim == 1:
    hmrad = hmrad.reshape(-1, 6)

# Doble máscara: grupo correcto + SubhaloFlag != 0
mask_fof  = all_subs['SubhaloGrNr'] == FOF_IDX
mask_flag = all_subs['SubhaloFlag'].astype(int) != 0
mask_sel  = mask_fof & mask_flag

sub_ids_sel   = all_sub_ids[mask_sel]
sub_pos_sel   = all_subs['SubhaloPos'][mask_sel]   * P.UL   # kpc
sub_mstar_sel = mtype[mask_sel, 4]                 * P.UM   # M_sun estrellas
sub_mtot_sel  = all_subs['SubhaloMass'][mask_sel]  * P.UM   # M_sun total
sub_rh_sel    = hmrad[mask_sel, 4]                 * P.UL   # kpc

n_fof   = int(mask_fof.sum())
n_sel   = int(mask_sel.sum())
BCG_sub = int(sub_ids_sel[0]) if n_sel > 0 else BCG_sub  # sub_id del BCG (primero del FoF)

print(f"Subhalos en el FoF {FOF_IDX}   : {n_fof}")
print(f"Con SubhaloFlag != 0           : {n_sel}")
print(f"BCG subfind ID (primero)       : {BCG_sub}")
print(f"\nRango masa estelar [log M_sun]:")
valid_m = sub_mstar_sel > 0
if valid_m.sum() > 0:
    print(f"  {np.log10(sub_mstar_sel[valid_m].min()):.2f} – "
          f"{np.log10(sub_mstar_sel[valid_m].max()):.2f}")

Subhalos en el FoF 0   : 17185
Con SubhaloFlag != 0           : 17066
BCG subfind ID (primero)       : 0

Rango masa estelar [log M_sun]:
  5.61 – 12.57


## Funciones auxiliares (idénticas a 01_ y 01b_)

In [ ]:
def rotate_by_inertia_tensor(pos_rel, mass, r_lim=np.inf):
    dist = np.linalg.norm(pos_rel, axis=1)
    ok   = (dist > 0) & (dist <= r_lim) & np.isfinite(mass)
    p, m = pos_rel[ok], mass[ok]
    if m.sum() == 0 or len(m) < 4:
        return pos_rel, np.eye(3)
    w    = 1.0 / dist[ok]**2
    mtot = np.sum(m)
    Ixx  = np.sum(m * p[:,0]**2 * w) / mtot
    Iyy  = np.sum(m * p[:,1]**2 * w) / mtot
    Izz  = np.sum(m * p[:,2]**2 * w) / mtot
    Ixy  = np.sum(m * p[:,0] * p[:,1] * w) / mtot
    Ixz  = np.sum(m * p[:,0] * p[:,2] * w) / mtot
    Iyz  = np.sum(m * p[:,1] * p[:,2] * w) / mtot
    I    = np.array([[Ixx, Ixy, Ixz],
                     [Ixy, Iyy, Iyz],
                     [Ixz, Iyz, Izz]])
    eigvals, eigvecs = np.linalg.eigh(I)
    idx    = np.argsort(eigvals)[::-1]
    R_mat  = eigvecs[:, idx].T
    return pos_rel @ R_mat.T, R_mat


def sb_profile_r(r_2d, lum_r, r_max_kpc, n_bins=60):
    r_bins = np.logspace(np.log10(0.5), np.log10(max(r_max_kpc, 1.0)), n_bins + 1)
    r_mid  = np.sqrt(r_bins[:-1] * r_bins[1:])
    mu_B   = np.full(n_bins, np.nan)
    for k, (r1, r2) in enumerate(zip(r_bins[:-1], r_bins[1:])):
        mk = (r_2d >= r1) & (r_2d < r2)
        if mk.sum() == 0:
            continue
        area_pc2 = np.pi * ((r2 * 1e3)**2 - (r1 * 1e3)**2)
        sigma_L  = lum_r[mk].sum() / area_pc2
        if sigma_L > 0:
            mu_B[k] = P.SB_CONST - 2.5 * np.log10(sigma_L)
    return r_mid, mu_B


def sb_profile_smooth(r_2d, lum_r, r_max_kpc, n_bins=60, frac_sigma=0.15):
    """
    Perfil 1D de brillo superficial suavizado con kernel gaussiano.

    A diferencia de sb_profile_r (anillos de bordes duros y disjuntos), aquí
    cada punto r_mid[k] pondera TODAS las partículas con un peso gaussiano
    centrado en r_mid[k] y ancho frac_sigma*r_mid[k] (fraccional, escala con
    el radio igual que el espaciado logarítmico de r_bins). Las ventanas de
    puntos vecinos se solapan, lo que promedia el ruido de Poisson entre
    ellos y suaviza la curva — a costa de perder resolución radial fina y de
    correlacionar puntos vecinos (dejan de ser estadísticamente independientes).

    frac_sigma : ancho del kernel como fracción del radio (0.15 = ±15%).
    """
    r_bins = np.logspace(np.log10(0.5), np.log10(max(r_max_kpc, 1.0)), n_bins + 1)
    r_mid  = np.sqrt(r_bins[:-1] * r_bins[1:])
    mu_B   = np.full(n_bins, np.nan)

    for k, r0 in enumerate(r_mid):
        sigma_k = frac_sigma * r0
        if sigma_k <= 0:
            continue
        w = np.exp(-0.5 * ((r_2d - r0) / sigma_k) ** 2)
        if w.sum() == 0:
            continue
        area_pc2 = 2 * np.pi * r0 * sigma_k * 1e6   # área efectiva del anillo gaussiano, kpc→pc
        sigma_L  = np.sum(lum_r * w) / area_pc2
        if sigma_L > 0:
            mu_B[k] = P.SB_CONST - 2.5 * np.log10(sigma_L)

    return r_mid, mu_B


def holmberg_radius(r_mid, mu_B, mu_cut=P.MU_HOLMBERG):
    valid = np.isfinite(mu_B) & (r_mid > 0)
    if valid.sum() < 3:
        return np.nan
    r_v, m_v = r_mid[valid], mu_B[valid]
    idx_s    = np.argsort(r_v)
    r_v, m_v = r_v[idx_s], m_v[idx_s]
    if m_v[0] > mu_cut or m_v[-1] < mu_cut:
        return np.nan
    try:
        f   = interp1d(m_v, r_v, kind='linear', fill_value='extrapolate')
        r_h = float(f(mu_cut))
        return r_h if 0 < r_h <= r_v[-1] * 1.2 else np.nan
    except Exception:
        return np.nan

## Ajuste del perfil de Holmberg a cada subhalo

Para cada subhalo con `SubhaloFlag != 0`:
1. Carga las partículas estelares con `loadSubhalo`
2. Centra en la posición del subhalo (no en el BCG)
3. Rota por tensor de inercia
4. Calcula `r_h` (radio de Holmberg)
5. Separa partículas **dentro** (`r_2d ≤ r_h`) y **fuera** (`r_2d > r_h`)

In [ ]:
fields_star = ['Coordinates', 'Masses', 'GFM_StellarPhotometrics']

results = []
outer_pos_list   = []
outer_mass_list  = []
outer_lum_list   = []
outer_subid_list = []

box_kpc = Header['BoxSize'] * P.UL

print(f"Procesando {n_sel} subhalos...")
for j, sid in enumerate(sub_ids_sel):
    try:
        st = il.snapshot.loadSubhalo(P.basePath, P.SNAP, int(sid), 'stars',
                                      fields=fields_star)
        if isinstance(st, int) or st is None or len(st['Masses']) == 0:
            results.append({'sub_id': sid, 'n_stars': 0, 'r_h': np.nan,
                            'n_inner': 0, 'n_outer': 0})
            continue

        pos_raw = st['Coordinates'] * P.UL
        mass    = st['Masses'] * P.UM
        phot    = st['GFM_StellarPhotometrics']
        if phot.ndim == 1:
            phot = phot.reshape(-1, 8)
        # índice 1 = B-band (Johnson, Vega) en GFM_StellarPhotometrics (U,B,V,K,g,r,i,z)
        lum_b = 10**((P.M_SUN_B_VEGA - phot[:, 1]) / 2.5)

        cen_sub = sub_pos_sel[j]
        pos_c   = Catalogue.Distance_3D(pos_raw, cen_sub, box_kpc)

        pos_rot, _ = rotate_by_inertia_tensor(pos_c, mass)
        r_2d = np.sqrt(pos_rot[:,0]**2 + pos_rot[:,1]**2)

        r_max       = np.percentile(r_2d, 99)
        # r_mid, mu_B = sb_profile_r(r_2d, lum_b, r_max)
        r_mid, mu_B = sb_profile_smooth(r_2d, lum_b, r_max)
        r_h         = holmberg_radius(r_mid, mu_B)

        if np.isfinite(r_h):
            mask_out = r_2d > r_h
        else:
            mask_out = np.zeros(len(mass), dtype=bool)

        pos_rel_bcg = Catalogue.Distance_3D(pos_raw[mask_out], GroupPos, box_kpc)
        outer_pos_list.append(pos_rel_bcg)
        outer_mass_list.append(mass[mask_out])
        outer_lum_list.append(lum_b[mask_out])
        outer_subid_list.append(np.full(mask_out.sum(), j))

        results.append({
            'sub_id'  : sid,
            'n_stars' : len(mass),
            'r_h'     : r_h,
            'n_inner' : int((~mask_out).sum()),
            'n_outer' : int(mask_out.sum()),
            'mstar'   : sub_mstar_sel[j],
            'r_mid'   : r_mid,
            'mu_B'    : mu_B,
            'cen'     : cen_sub,
        })

    except Exception as e:
        print(f"  [WARN] sub_id={sid}: {e}")
        results.append({'sub_id': sid, 'n_stars': 0, 'r_h': np.nan,
                        'n_inner': 0, 'n_outer': 0})

    if (j+1) % 5 == 0 or j+1 == n_sel:
        print(f"  {j+1}/{n_sel}", end="\r")

if outer_pos_list:
    outer_pos   = np.concatenate(outer_pos_list,   axis=0)
    outer_mass  = np.concatenate(outer_mass_list)
    outer_lum   = np.concatenate(outer_lum_list)
    outer_subid = np.concatenate(outer_subid_list).astype(int)
else:
    outer_pos = outer_mass = outer_lum = outer_subid = np.array([])

n_outer_total = len(outer_mass)
r_h_vals  = np.array([r['r_h'] for r in results])
n_with_rh = np.sum(np.isfinite(r_h_vals))

print(f"\nResultados:")
print(f"  Subhalos con r_h válido : {n_with_rh} / {n_sel}")
print(f"  Partículas exteriores   : {n_outer_total:,}")

## Tabla resumen de subhalos

In [ ]:
import pandas as pd

rows = []
for r in results:
    mstar = r.get('mstar', 0)
    rows.append({
        'sub_id'     : r['sub_id'],
        'es_BCG'     : '★' if r['sub_id'] == BCG_sub else '',
        'log_Mstar'  : f"{np.log10(mstar):.2f}" if mstar > 0 else '—',
        'N_estrellas': r['n_stars'],
        'r_h [kpc]'  : f"{r['r_h']:.1f}" if np.isfinite(r['r_h']) else '—',
        'N_interior' : r['n_inner'],
        'N_exterior' : r['n_outer'],
        'f_exterior' : f"{r['n_outer']/r['n_stars']:.3f}" if r['n_stars'] > 0 else '—',
    })

df = pd.DataFrame(rows)
display(df.head(5).style.background_gradient(subset=['N_exterior'], cmap='Reds'))

## Figura 1 – Perfiles de Holmberg individuales

Un panel por subhalo con perfil válido (máximo 12 para no saturar la figura).  
El BCG se muestra siempre en el primer panel.

In [ ]:
# Seleccionar subhalos con perfil válido, BCG primero
valid_results = [r for r in results if np.isfinite(r.get('r_h', np.nan)) and r['n_stars'] > 0]
bcg_res  = [r for r in valid_results if r['sub_id'] == BCG_sub]
rest_res = [r for r in valid_results if r['sub_id'] != BCG_sub]
plot_res = (bcg_res + rest_res)[:12]   # máximo 12 paneles

n_plot = len(plot_res)
if n_plot == 0:
    print("Sin subhalos con perfil válido para graficar.")
else:
    ncols = min(4, n_plot)
    nrows = int(np.ceil(n_plot / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 4*nrows))
    axes = np.array(axes).flatten()

    for ax, r in zip(axes, plot_res):
        v = np.isfinite(r['mu_B'])
        label = f"sub {r['sub_id']}"
        if r['sub_id'] == BCG_sub:
            label += " ★ BCG"
        color = 'royalblue' if r['sub_id'] == BCG_sub else 'dimgray'

        ax.plot(r['r_mid'][v], r['mu_B'][v], '-', color=color, lw=1.8)
        ax.axhline(P.MU_HOLMBERG, color='r', ls='--', lw=1.2,
                   label=f'μ={P.MU_HOLMBERG}')
        ax.axvline(r['r_h'], color='r', ls=':', lw=1.2,
                   label=f"r_h={r['r_h']:.0f} kpc")
        ax.fill_betweenx([15, P.MU_HOLMBERG + 3],
                          0, r['r_h'], alpha=0.08, color='royalblue')
        ax.fill_betweenx([15, P.MU_HOLMBERG + 3],
                          r['r_h'], r['r_mid'][v][-1], alpha=0.08, color='tomato')
        ax.invert_yaxis()
        ax.set_ylim(35, 16)
        ax.set_title(label, fontsize=10)
        ax.set_xlabel('r [kpc]', fontsize=9)
        ax.set_ylabel(r'$\mu_B$', fontsize=9)
        ax.legend(fontsize=7, loc='lower right')

    # Apagar paneles vacíos
    for ax in axes[n_plot:]:
        ax.set_visible(False)

    plt.suptitle(f'Perfiles de Holmberg — FoF {FOF_IDX}  (log M200c={np.log10(M200c):.2f})',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIG_PDF}/fig01c_perfiles_holmberg_fof{FOF_IDX}.pdf', bbox_inches='tight')
    plt.savefig(f'{FIG_PNG}/fig01c_perfiles_holmberg_fof{FOF_IDX}.png', bbox_inches='tight', dpi=150)
    plt.show()

## Figura 2 – Mapa 2D de partículas exteriores (envolvente ICL)

Proyección en el plano xy centrada en el BCG. Se muestran **solo las partículas fuera del radio de Holmberg** de su subhalo respectivo. Cada subhalo tiene un color distinto para identificar de dónde vienen.

In [ ]:
if len(outer_mass) == 0:
    print("Sin partículas exteriores para graficar.")
else:
    # Proyección 2D (plano xy) — ya están en coordenadas relativas al BCG
    x_out = outer_pos[:, 0]
    y_out = outer_pos[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))

    # — Panel izquierdo: coloreado por subhalo de origen —
    ax = axes[0]
    cmap_sub = plt.cm.get_cmap('tab20', n_sel)
    sc = ax.scatter(x_out, y_out, c=outer_subid, cmap=cmap_sub,
                    s=0.3, alpha=0.4, rasterized=True)
    ax.add_patch(plt.Circle((0, 0), R200c, fill=False, color='white',
                              lw=1.5, ls='--', label=f'R₂₀₀c={R200c:.0f} kpc'))
    # Marcar centros de subhalos seleccionados
    for j, r in enumerate(results):
        if r['n_stars'] > 0:
            cen_rel = Catalogue.Distance_3D(
                r['cen'].reshape(1,3), GroupPos, box_kpc)[0]
            ax.plot(cen_rel[0], cen_rel[1], '+',
                    color=cmap_sub(j), ms=6, mew=1.2)
    ax.set_xlim(-R200c*1.1, R200c*1.1)
    ax.set_ylim(-R200c*1.1, R200c*1.1)
    ax.set_facecolor('black')
    ax.set_xlabel('x [kpc]'); ax.set_ylabel('y [kpc]')
    ax.set_title('Partículas exteriores — color por subhalo de origen')
    ax.legend(fontsize=9, loc='upper right')

    # — Panel derecho: brillo superficial acumulado —
    ax = axes[1]
    r_plot  = R200c
    n_pix   = 256
    edges   = np.linspace(-r_plot, r_plot, n_pix + 1)
    pix_pc2 = ((2 * r_plot / n_pix) * 1e3)**2

    from scipy.stats import binned_statistic_2d
    H, _, _, _ = binned_statistic_2d(x_out, y_out, outer_lum,
                                      statistic='sum', bins=[edges, edges])
    with np.errstate(divide='ignore', invalid='ignore'):
        sigma  = np.where(H > 0, H / pix_pc2, np.nan)
        mu_map = np.where(sigma > 0, P.SB_CONST - 2.5 * np.log10(sigma), np.nan)

    im = ax.imshow(mu_map.T, origin='lower', cmap='inferno_r',
                   extent=[-r_plot, r_plot, -r_plot, r_plot],
                   vmin=24, vmax=32)
    plt.colorbar(im, ax=ax, label=r'$\mu_B$ [mag arcsec$^{-2}$]')
    ax.add_patch(plt.Circle((0, 0), R200c, fill=False, color='white',
                              lw=1.5, ls='--', label=f'R₂₀₀c'))
    ax.set_xlabel('x [kpc]'); ax.set_ylabel('y [kpc]')
    ax.set_title('Brillo superficial — envolvente exterior')
    ax.legend(fontsize=9)

    plt.suptitle(f'Partículas exteriores al radio de Holmberg — FoF {FOF_IDX}',
                 fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{FIG_PDF}/fig01c_mapa_exterior_fof{FOF_IDX}.pdf', bbox_inches='tight')
    plt.savefig(f'{FIG_PNG}/fig01c_mapa_exterior_fof{FOF_IDX}.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Total partículas graficadas: {len(outer_mass):,}")

## Figura 3 – Perfil de brillo superficial acumulado de la envolvente

Suma las luminosidades de TODAS las partículas exteriores en anillos concéntricos centrados en el BCG. Equivale al perfil ICL global del grupo.

In [ ]:
if len(outer_mass) > 0:
    r_2d_out = np.sqrt(outer_pos[:,0]**2 + outer_pos[:,1]**2)
    r_max_out = np.percentile(r_2d_out, 99)
    # r_mid_icl, mu_icl = sb_profile_r(r_2d_out, outer_lum, r_max_out, n_bins=50)
    r_mid_icl, mu_icl = sb_profile_smooth(r_2d_out, outer_lum, r_max_out, n_bins=50)

    fig, ax = plt.subplots(figsize=(8, 5))
    v = np.isfinite(mu_icl)
    ax.plot(r_mid_icl[v], mu_icl[v], 'k-', lw=2,
            label='Envolvente exterior (todas las galaxias)')
    ax.axhline(P.MU_HOLMBERG, color='r', ls='--', lw=1.5,
               label=f'μ_r = {P.MU_HOLMBERG} mag/arcsec²')
    ax.axvline(R200c, color='gray', ls='-.', lw=1.2,
               label=f'R₂₀₀c = {R200c:.0f} kpc')
    ax.invert_yaxis()
    ax.set_xlabel('Radio desde el BCG [kpc]')
    ax.set_ylabel(r'$\mu_B$ [mag arcsec$^{-2}$]')
    ax.set_title(f'Perfil SB acumulado — envolvente exterior\nFoF {FOF_IDX}  (log M200c={np.log10(M200c):.2f})')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{FIG_PDF}/fig01c_perfil_envolvente_fof{FOF_IDX}.pdf', bbox_inches='tight')
    plt.savefig(f'{FIG_PNG}/fig01c_perfil_envolvente_fof{FOF_IDX}.png', bbox_inches='tight', dpi=150)
    plt.show()

## Figura 4 – Masa exterior vs masa estelar del subhalo

¿Los subhalos más masivos tienen más partículas en su envolvente, o la fracción exterior es constante?